# Recomendacion basada en embeddings semanticos

Este notebook recoge el siguiente paso metodologico tras el baseline del notebook 01. Una vez visto que la clasificacion supervisada clasica funciona bien pero depende fuertemente del vocabulario observado, aqui se pasa de clasificacion a recomendacion semantica usando embeddings.

En lugar de entrenar un clasificador para asignar una unica categoria, cada publicacion se representa mediante un embedding semantico y las recomendaciones se obtienen comparando la cercania entre un perfil de usuario y los posts disponibles. La idea central es reutilizar el mismo dataset textual y sustituir la logica de prediccion por una logica de similitud.

In [9]:
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError(
        "Falta instalar sentence-transformers. Ejecuta: %pip install -q sentence-transformers"
    ) from exc

DATA_PATH = Path("data/posts.csv")
EXPECTED_COLUMNS = ["id", "descripcion", "hashtags", "categoria"]
MODELS_DIR = Path("models")
ARTIFACTS_DIR = Path("artifacts")
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_MODEL_DIR = MODELS_DIR / "modelo_embeddings"
EMBEDDING_METADATA_PATH = MODELS_DIR / "clasificador_embeddings.joblib"
EMBEDDINGS_ARTIFACT_PATH = ARTIFACTS_DIR / "embeddings_posts.joblib"
TOP_K = 5
INTERESES_USUARIO = ["playa", "viajes"]


## 1. Carga del dataset textual

Se utiliza el mismo archivo `data/posts.csv` del notebook anterior. Igual que en el experimento de clasificacion, la señal textual principal se construye uniendo `descripcion` y `hashtags`.

In [10]:
df = pd.read_csv(DATA_PATH, encoding="utf-8")

actual_columns = list(df.columns)
if set(actual_columns) != set(EXPECTED_COLUMNS):
    raise ValueError(
        "El CSV no tiene el esquema esperado. "
        f"Columnas esperadas: {EXPECTED_COLUMNS}. "
        f"Columnas encontradas: {actual_columns}."
    )

df = df[EXPECTED_COLUMNS].copy()

df["texto_original"] = (
    df["descripcion"].fillna("").astype(str).str.strip()
    + " "
    + df["hashtags"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

print(f"Archivo cargado desde: {DATA_PATH}")
print(f"Numero total de registros: {len(df)}")
display(df.head())


Archivo cargado desde: data\posts.csv
Numero total de registros: 5000


,id,descripcion,hashtags,categoria,texto_original
0,1,"temazo, descubriendo joyitas indies en spotify...",#Acustico #Indie,musica,"temazo, descubriendo joyitas indies en spotify..."
1,2,este gato callejero ya es prácticamente mío qu...,#AmorAnimal #DogLife #Naturaleza #Veterinario,animales,este gato callejero ya es prácticamente mío qu...
2,3,Literalmente el mejor momento del día: este ga...,NaN,animales,Literalmente el mejor momento del día: este ga...
3,4,"Perdidos por Cádiz, perdido en el metro de Rep...",#explorando,viajes,"Perdidos por Cádiz, perdido en el metro de Rep..."
4,5,Próxima parada: pagando un café a precio de tu...,#Paisaje #Naturaleza #Mochileros #RoadTrip #Tr...,viajes,Próxima parada: pagando un café a precio de tu...


## 2. Preparacion del texto

Para mantener comparabilidad con el notebook 01, se reutiliza la misma idea de preprocesamiento sobre `descripcion + hashtags`. Aunque los embeddings suelen tolerar texto mas natural que TF-IDF, conservar este paso permite trabajar exactamente con la misma senal textual ya preparada.


In [11]:
def preprocesar_texto(texto: str) -> str:
    texto = str(texto).lower()
    texto = texto.replace("#", " ")
    texto = re.sub(r"[^a-zA-Z0-9\sáéíóúüñ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


df["texto_limpio"] = df["texto_original"].apply(preprocesar_texto)
df_modelo = df[df["texto_limpio"].str.len() > 0].copy().reset_index(drop=True)

print(f"Registros originales: {len(df)}")
print(f"Registros con texto util: {len(df_modelo)}")
print(f"Registros excluidos por texto vacio: {len(df) - len(df_modelo)}")
display(df_modelo[["descripcion", "hashtags", "categoria", "texto_limpio"]].head(8))


Registros originales: 5000
Registros con texto util: 4931
Registros excluidos por texto vacio: 69


,descripcion,hashtags,categoria,texto_limpio
0,"temazo, descubriendo joyitas indies en spotify...",#Acustico #Indie,musica,temazo descubriendo joyitas indies en spotify ...
1,este gato callejero ya es prácticamente mío qu...,#AmorAnimal #DogLife #Naturaleza #Veterinario,animales,este gato callejero ya es prácticamente mío qu...
2,Literalmente el mejor momento del día: este ga...,NaN,animales,literalmente el mejor momento del día este gat...
3,"Perdidos por Cádiz, perdido en el metro de Rep...",#explorando,viajes,perdidos por cádiz perdido en el metro de repú...
4,Próxima parada: pagando un café a precio de tu...,#Paisaje #Naturaleza #Mochileros #RoadTrip #Tr...,viajes,próxima parada pagando un café a precio de tur...
5,"Que ternura, mi perro corriendo como un loco p...",#mascotas#animallover#michis,animales,que ternura mi perro corriendo como un loco po...
6,"brum brum, la gasolina a precio de oooooro per...",#carporn #tuning #llantas #exhaust #v8,coches,brum brum la gasolina a precio de oooooro pero...
7,"perdido en el metro de Sierra Leona, ayuda me ...",#roadtrip#paisaje#escapada,viajes,perdido en el metro de sierra leona ayuda me m...


## 3. Generacion y persistencia de embeddings

Un **embedding** es un vector numerico denso que intenta resumir el significado de un texto dentro de un espacio semantico. En ese espacio, dos frases cercanas suelen compartir tema o intencion, incluso si no usan exactamente las mismas palabras.

En este notebook se importa el modelo `SentenceTransformer`, se guarda localmente en `models/modelo_embeddings` y se persisten los embeddings de los posts en `artifacts/embeddings_posts.joblib`. Asi, el notebook 03 puede reutilizar exactamente la misma representacion sin depender de variables creadas solo en memoria.


In [12]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if EMBEDDING_MODEL_DIR.exists():
    modelo_embeddings = SentenceTransformer(str(EMBEDDING_MODEL_DIR), local_files_only=True)
    origen_modelo_embeddings = f"modelo local persistido en {EMBEDDING_MODEL_DIR}"
else:
    modelo_embeddings = SentenceTransformer(EMBEDDING_MODEL_NAME)
    modelo_embeddings.save(str(EMBEDDING_MODEL_DIR))
    origen_modelo_embeddings = f"modelo descargado/importado desde {EMBEDDING_MODEL_NAME} y guardado en {EMBEDDING_MODEL_DIR}"

embeddings_posts = modelo_embeddings.encode(
    df_modelo["texto_limpio"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

artefacto_embeddings = {
    "embeddings": embeddings_posts.astype(np.float32),
    "posts": df_modelo[["id", "descripcion", "hashtags", "categoria", "texto_original", "texto_limpio"]].copy(),
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "embedding_model_dir": str(EMBEDDING_MODEL_DIR),
    "text_column": "texto_limpio",
    "data_path": str(DATA_PATH),
}

metadata_embeddings = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "embedding_model_dir": str(EMBEDDING_MODEL_DIR),
    "embeddings_artifact_path": str(EMBEDDINGS_ARTIFACT_PATH),
    "text_column": "texto_limpio",
    "data_path": str(DATA_PATH),
}

joblib.dump(artefacto_embeddings, EMBEDDINGS_ARTIFACT_PATH)
joblib.dump(metadata_embeddings, EMBEDDING_METADATA_PATH)

print(f"Modelo de embeddings: {EMBEDDING_MODEL_NAME}")
print(f"Origen del modelo: {origen_modelo_embeddings}")
print(f"Matriz de embeddings: {embeddings_posts.shape[0]} posts x {embeddings_posts.shape[1]} dimensiones")
print(f"Artefacto de embeddings guardado en: {EMBEDDINGS_ARTIFACT_PATH}")
print(f"Metadatos del modelo guardados en: {EMBEDDING_METADATA_PATH}")
print("Primeras 5 componentes del primer embedding:")
print(np.round(embeddings_posts[0][:5], 4))


Batches: 100%|██████████| 78/78 [00:40<00:00,  1.92it/s]

Modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Origen del modelo: modelo local persistido en models\modelo_embeddings
Matriz de embeddings: 4931 posts x 384 dimensiones
Artefacto de embeddings guardado en: artifacts\embeddings_posts.joblib
Metadatos del modelo guardados en: models\clasificador_embeddings.joblib
Primeras 5 componentes del primer embedding:
[ 0.034  -0.0753  0.0614 -0.0207 -0.0103]


## 4. Simulacion de un perfil de usuario

En un recomendador basado en contenido, el **perfil de usuario** es una representacion vectorial de los intereses del usuario. Aqui se simula un usuario con afinidad por las categorias `playa` y `viajes`.

Para construir su perfil, se genera un embedding para cada interes y despues se calcula el promedio. Ese promedio actua como un centro semantico de preferencias: cuanto mas cerca este un post de ese vector medio, mas afin se considera respecto al usuario.

In [13]:
embeddings_intereses = modelo_embeddings.encode(
    INTERESES_USUARIO,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

perfil_usuario = embeddings_intereses.mean(axis=0, keepdims=True)
perfil_usuario = perfil_usuario / np.linalg.norm(perfil_usuario, axis=1, keepdims=True)

print(f"Intereses del usuario: {INTERESES_USUARIO}")
print(f"Shape del perfil de usuario: {perfil_usuario.shape}")


Intereses del usuario: ['playa', 'viajes']
Shape del perfil de usuario: (1, 384)


## 5. Similitud coseno y recuperacion de recomendaciones

La **similitud coseno** mide cuanto se parecen dos vectores segun su orientacion. Un valor cercano a `1` indica gran parecido semantico; un valor cercano a `0` indica poca relacion. En recomendacion textual, esto permite ordenar los posts segun su proximidad al perfil del usuario.

No hay entrenamiento supervisado, no hay etiquetas objetivo y no hay division `train/test`: el sistema solo compara representaciones semanticas ya calculadas.

In [14]:
similitudes = cosine_similarity(perfil_usuario, embeddings_posts).ravel()

recomendaciones = df_modelo[["id", "descripcion", "hashtags", "categoria", "texto_limpio"]].copy()
recomendaciones["score_similitud"] = similitudes

top_k_posts = recomendaciones.sort_values("score_similitud", ascending=False).head(TOP_K).copy()
top_k_posts["score_similitud"] = top_k_posts["score_similitud"].round(4)

display(top_k_posts[["score_similitud", "categoria", "descripcion", "hashtags", "texto_limpio"]])


,score_similitud,categoria,descripcion,hashtags,texto_limpio
1319,0.8856,viajes,NaN,#Trip,trip
4916,0.8740,playa,NaN,#playa,playa
4287,0.8639,viajes,NaN,#travel,travel
4622,0.8505,playa,NaN,#mar #vacaciones,mar vacaciones
2514,0.8452,viajes,NaN,#viajero,viajero


In [15]:
for posicion, (_, fila) in enumerate(top_k_posts.iterrows(), start=1):
    print(f"Top {posicion} | score={fila['score_similitud']}")
    print(f"Categoria: {fila['categoria']}")
    print(f"Texto recomendado: {fila['texto_limpio']}")
    print("-" * 100)


Top 1 | score=0.8855999708175659
Categoria: viajes
Texto recomendado: trip
----------------------------------------------------------------------------------------------------
Top 2 | score=0.8740000128746033
Categoria: playa
Texto recomendado: playa
----------------------------------------------------------------------------------------------------
Top 3 | score=0.8639000058174133
Categoria: viajes
Texto recomendado: travel
----------------------------------------------------------------------------------------------------
Top 4 | score=0.8504999876022339
Categoria: playa
Texto recomendado: mar vacaciones
----------------------------------------------------------------------------------------------------
Top 5 | score=0.8452000021934509
Categoria: viajes
Texto recomendado: viajero
----------------------------------------------------------------------------------------------------


## 6. Evaluacion basica del ranking con Precision@K

Aunque este notebook no trabaja con un conjunto de prueba supervisado, si puede hacerse una evaluacion sencilla del ranking recuperado. Para esta simulacion, se consideran relevantes las categorias coherentes con el perfil del usuario, es decir, `playa` y `viajes`.

`Precision@K` mide la proporcion de elementos relevantes dentro de las primeras `K` recomendaciones. No evalua clasificacion, sino calidad de recuperacion en las primeras posiciones del ranking.

In [16]:
CATEGORIAS_RELEVANTES = {"playa", "viajes"}

def precision_at_k(recomendaciones_df, categorias_relevantes):
    return recomendaciones_df["categoria"].isin(categorias_relevantes).mean()

resultados_precision = []
for k in [3, 5]:
    top_k_eval = recomendaciones.sort_values("score_similitud", ascending=False).head(k).copy()
    resultados_precision.append({
        "k": k,
        "precision_at_k": round(precision_at_k(top_k_eval, CATEGORIAS_RELEVANTES), 4),
        "categorias_top_k": " | ".join(top_k_eval["categoria"].tolist()),
    })

df_precision = pd.DataFrame(resultados_precision)
display(df_precision)


,k,precision_at_k,categorias_top_k
0,3,1.0,viajes | playa | viajes
1,5,1.0,viajes | playa | viajes | playa | viajes


## 7. Por que este enfoque es mas flexible que TF-IDF + LogisticRegression

El enfoque basado en embeddings es mas flexible por varias razones. Primero, no depende de entrenar un clasificador supervisado ni de disponer de etiquetas para cada nueva necesidad de recomendacion. Segundo, puede relacionar textos por significado semantico, aunque no compartan exactamente las mismas palabras. Tercero, produce un ranking continuo de afinidad en lugar de forzar una unica categoria como salida.

Frente al pipeline `TF-IDF + LogisticRegression`, este sistema puede adaptarse mejor a sinonimos, reformulaciones, nombres propios o expresiones nuevas que no estaban presentes en el vocabulario de entrenamiento. En otras palabras, mientras TF-IDF se apoya sobre todo en coincidencias lexicas, los embeddings permiten recomendar por cercania de significado, lo que resulta mas natural para un sistema de recomendacion basado en contenido.

Metodologicamente, este notebook no sustituye al anterior: lo complementa. El notebook 01 demuestra lo que puede hacer una clasificacion supervisada clasica; este notebook muestra por que, cuando el objetivo pasa de etiquetar a recomendar, una representacion semantica resulta mas adecuada.